In [1]:
import pandas as pd

sessions = pd.read_csv(
    "output_data/hmog_raw_feta_audit/sessions.csv"
)

session = sessions[
    (sessions["user_id"].astype(str) == "180679") &
    (sessions["session_number"].astype(str) == "12")
].iloc[0]

gestures = pd.read_csv(session["recovered_gestures_file"])

swipes = gestures[
    (gestures["type"] == "swipe") &
    (gestures["portrait_single_valid"] == 1)
]

failed = swipes[swipes["both_strict_paired"] == 0]

reasons = {
    "Fewer than 3 sensor samples": (
        (failed["accel_samples"] < 3) |
        (failed["gyro_samples"] < 3)
    ),
    "Sensor edge gap > 40 ms": (
        (failed["accel_edge_max_ms"] > 40) |
        (failed["gyro_edge_max_ms"] > 40)
    ),
    "Internal sensor gap > 120 ms": (
        (failed["accel_max_gap_ms"] > 120) |
        (failed["gyro_max_gap_ms"] > 120)
    ),
    "No accel/gyro pair within 25 ms": (
        failed["accel_gyro_pairs"] < 1
    ),
}

print(f"Portrait swipes: {len(swipes)}")
print(f"Failed swipes: {len(failed)}")

for name, condition in reasons.items():
    print(f"{name}: {condition.sum()}")

display(
    failed[[
        "duration_ms",
        "accel_samples",
        "gyro_samples",
        "accel_edge_max_ms",
        "gyro_edge_max_ms",
        "accel_max_gap_ms",
        "gyro_max_gap_ms",
        "accel_gyro_pairs",
    ]]
)

Portrait swipes: 28
Failed swipes: 28
Fewer than 3 sensor samples: 0
Sensor edge gap > 40 ms: 17
Internal sensor gap > 120 ms: 27
No accel/gyro pair within 25 ms: 0


,duration_ms,accel_samples,gyro_samples,accel_edge_max_ms,gyro_edge_max_ms,accel_max_gap_ms,gyro_max_gap_ms,accel_gyro_pairs
0,34342.0,3394,3394,63.5,61.5,143.0,168.0,3392
1,6516.0,644,643,7.5,8.5,134.0,135.0,643
2,10365.0,1029,1029,44.5,42.5,142.0,142.0,1028
3,3326.0,327,327,14.5,13.5,147.0,150.0,327
4,623.0,60,60,59.5,58.5,124.0,124.0,60
5,1026.0,101,102,29.5,28.5,122.0,123.0,101
6,327.0,26,26,73.5,72.5,124.0,123.0,26
7,5378.0,533,533,1.5,1.5,151.0,151.0,533
8,1993.0,192,192,94.5,95.5,115.0,128.0,192
9,956.0,97,96,24.5,25.5,121.0,122.0,96


In [3]:
import pandas as pd

results = []

for max_edge_gap in [40, 60, 80, 120]:
    for max_internal_gap in [120, 140, 160, 180, 200]:

        covered = (
            (swipes["accel_samples"] >= 3) &
            (swipes["gyro_samples"] >= 3) &
            (swipes["accel_edge_max_ms"] <= max_edge_gap) &
            (swipes["gyro_edge_max_ms"] <= max_edge_gap) &
            (swipes["accel_max_gap_ms"] <= max_internal_gap) &
            (swipes["gyro_max_gap_ms"] <= max_internal_gap) &
            (swipes["accel_gyro_pairs"] >= 1)
        )

        results.append({
            "max_edge_gap_ms": max_edge_gap,
            "max_internal_gap_ms": max_internal_gap,
            "covered_swipes": int(covered.sum()),
            "coverage_pct": round(covered.mean() * 100, 2),
        })

df = pd.DataFrame(results)

display(
    df.pivot(
        index="max_edge_gap_ms",
        columns="max_internal_gap_ms",
        values="coverage_pct"
    )
)

display(df)

max_internal_gap_ms,120,140,160,180,200
max_edge_gap_ms,,,,,
40,0.00,32.14,39.29,39.29,39.29
60,0.00,42.86,53.57,53.57,53.57
80,0.00,57.14,67.86,75.00,75.00
120,3.57,75.00,89.29,96.43,96.43


,max_edge_gap_ms,max_internal_gap_ms,covered_swipes,coverage_pct
0,40,120,0,0.00
1,40,140,9,32.14
2,40,160,11,39.29
3,40,180,11,39.29
4,40,200,11,39.29
5,60,120,0,0.00
6,60,140,12,42.86
7,60,160,15,53.57
8,60,180,15,53.57
9,60,200,15,53.57


In [4]:
from pathlib import Path
import pandas as pd
from analyze_hmog_portrait_selection_v1_5 import read_table

OUT = Path("output_data/hmog_portrait_selection_v1_5")

diag = pd.read_csv(
    OUT / "key_matching_diagnostics.csv",
    dtype={"user_id": str, "session_number": str}
)

sessions = pd.read_csv(
    OUT / "sessions.csv",
    dtype={"user_id": str, "session_number": str}
)

diag["extra_up"] = diag["key_up_rows"] - diag["key_down_rows"]

worst = diag.loc[diag["extra_up"].idxmax()]

session = sessions[
    (sessions["user_id"] == worst["user_id"]) &
    (sessions["session_number"] == worst["session_number"])
].iloc[0]

path = Path(session["session_dir"]) / "KeyPressEvent.csv"
keys = read_table(path, "key")

keys["presstype"] = pd.to_numeric(
    keys["presstype"], errors="coerce"
).astype("Int64")

table = (
    keys.groupby(["keyid", "presstype"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "DOWN", 1: "UP"})
)

table["extra_up"] = (
    table.get("UP", 0) - table.get("DOWN", 0)
)

print("User:", worst["user_id"])
print("Session:", worst["session_number"])
print("Extra UP:", worst["extra_up"])

display(table.sort_values("extra_up", ascending=False).head(30))

User: 918136
Session: 19
Extra UP: 857


presstype,DOWN,UP,extra_up
keyid,,,
32,159,288,129
116,41,121,80
97,26,99,73
111,42,112,70
101,55,114,59
110,43,91,48
104,17,65,48
105,25,64,39
115,34,67,33


In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
from analyze_hmog_portrait_selection_v1_5 import read_table

folder = path.parent

touch_path = next(
    p for p in (
        folder / "TouchEvent_im.csv",
        folder / "TouchEvent.csv",
    )
    if p.exists()
)

touch = read_table(touch_path, "touch")
keys = read_table(folder / "KeyPressEvent.csv", "key")

touch["action"] = pd.to_numeric(
    touch["actionid"], errors="coerce"
)
touch["time"] = pd.to_numeric(
    touch["eventtime"], errors="coerce"
).round(3)

keys["type"] = pd.to_numeric(
    keys["presstype"], errors="coerce"
)
keys["time"] = pd.to_numeric(
    keys["presstime"], errors="coerce"
).round(3)

def count_exact_matches(key_type, touch_action):
    k = (
        keys.loc[keys["type"] == key_type]
        .groupby(["activityid", "time"])
        .size()
    )
    t = (
        touch.loc[touch["action"] == touch_action]
        .groupby(["activityid", "time"])
        .size()
    )

    common = k.index.intersection(t.index)
    return int(np.minimum(
        k.loc[common].to_numpy(),
        t.loc[common].to_numpy()
    ).sum())

up = keys[keys["type"] == 1]

duplicate_ups = (
    up.groupby(["activityid", "time", "keyid"])
    .size()
    .sub(1)
    .clip(lower=0)
    .sum()
)

results = pd.Series({
    "Touch DOWN": int((touch["action"] == 0).sum()),
    "Touch UP": int((touch["action"] == 1).sum()),
    "Key DOWN": int((keys["type"] == 0).sum()),
    "Key UP": len(up),
    "Key DOWN matching Touch DOWN exactly": count_exact_matches(0, 0),
    "Key UP matching Touch UP exactly": count_exact_matches(1, 1),
    "Repeated Key UP (same key and timestamp)": int(duplicate_ups),
})

display(results.to_frame("count"))

,count
Touch DOWN,1759
Touch UP,1757
Key DOWN,825
Key UP,1682
Key DOWN matching Touch DOWN exactly,811
Key UP matching Touch UP exactly,1652
Repeated Key UP (same key and timestamp),0


In [7]:

from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from IPython.display import display

from analyze_hmog_portrait_selection_v1_5 import read_table

# User 918136, session 19
session_dir = Path(path).parent

touch_path = next(
    p for p in [
        session_dir / "TouchEvent_im.csv",
        session_dir / "TouchEvent.csv",
    ]
    if p.exists()
)

touch = read_table(touch_path, "touch").copy()
keys = read_table(session_dir / "KeyPressEvent.csv", "key").copy()

# 1. Prepare event timestamps and identifiers.
touch["act"] = touch["activityid"].astype(str).str.strip()
touch["ms"] = pd.to_numeric(
    touch["eventtime"], errors="coerce"
).round(3)
touch["action"] = pd.to_numeric(
    touch["actionid"], errors="coerce"
)
touch["pcount"] = pd.to_numeric(
    touch["pointercount"], errors="coerce"
)
touch["pid"] = pd.to_numeric(
    touch["pointerid"], errors="coerce"
)

keys["act"] = keys["activityid"].astype(str).str.strip()
keys["ms"] = pd.to_numeric(
    keys["presstime"], errors="coerce"
).round(3)
keys["type"] = pd.to_numeric(
    keys["presstype"], errors="coerce"
)
keys["kid"] = (
    pd.to_numeric(keys["keyid"], errors="coerce")
    .astype("Int64").astype(str)
)

# 2. Reconstruct completed physical contacts.
# Only simple, single-pointer DOWN -> UP contacts.
contacts = []

for act, group in touch.groupby("act", sort=False):
    down = None

    for row in group.sort_values("ms", kind="stable").itertuples():
        if not np.isfinite(row.ms):
            continue

        if row.pcount != 1 or row.pid != 0:
            down = None
            continue

        if row.action == 0:  # ACTION_DOWN
            down = float(row.ms)

        elif row.action == 1:  # ACTION_UP
            if down is not None and row.ms > down:
                contacts.append({
                    "act": act,
                    "down_ms": down,
                    "up_ms": float(row.ms),
                    "duration_ms": float(row.ms - down),
                })
            down = None

        elif row.action in (3, 5, 6):
            down = None  # CANCEL / pointer changes

contacts = pd.DataFrame(contacts)

if contacts.empty:
    raise ValueError("No completed single-pointer contacts found")

# 3. Match Key UP to Touch UP of a completed contact.
# cumcount makes matching one-to-one even if timestamps repeat.
contacts["nth"] = contacts.groupby(
    ["act", "up_ms"]
).cumcount()

ups = keys[keys["type"] == 1].copy()
downs = keys[keys["type"] == 0].copy()

ups["nth"] = ups.groupby(["act", "ms"]).cumcount()

matched = ups.merge(
    contacts,
    left_on=["act", "ms", "nth"],
    right_on=["act", "up_ms", "nth"],
    how="left",
    validate="one_to_one",
)

matched = matched.dropna(subset=["down_ms"]).copy()

# 4. Check whether the SAME physical contact has a
# Key DOWN with the SAME KeyID.

exact_downs = set(
    zip(downs["act"], downs["ms"], downs["kid"])
)
any_exact_downs = set(
    zip(downs["act"], downs["ms"])
)

matched["key_down_exact"] = [
    (act, start, kid) in exact_downs
    for act, start, kid in zip(
        matched["act"], matched["down_ms"], matched["kid"]
    )
]

matched["any_key_down_exact"] = [
    (act, start) in any_exact_downs
    for act, start in zip(
        matched["act"], matched["down_ms"]
    )
]

# Also test whether the same KeyID has a DOWN
# anywhere during the physical contact.
down_times = {
    key: np.sort(g["ms"].dropna().to_numpy(dtype=float))
    for key, g in downs.groupby(["act", "kid"])
}

def has_down_during_contact(row, tolerance_ms=0):
    times = down_times.get((row["act"], row["kid"]))
    if times is None or len(times) == 0:
        return False

    left = np.searchsorted(
        times, row["down_ms"] - tolerance_ms, side="left"
    )
    right = np.searchsorted(
        times, row["up_ms"] + tolerance_ms, side="right"
    )
    return right > left

matched["key_down_inside"] = matched.apply(
    has_down_during_contact, axis=1
)

matched["key_down_near_10ms"] = matched.apply(
    has_down_during_contact, axis=1,
    tolerance_ms=10
)

# 5. Summary.
summary = pd.Series({
    "Touch DOWN (raw)": int((touch["action"] == 0).sum()),
    "Touch UP (raw)": int((touch["action"] == 1).sum()),
    "Completed single-pointer contacts": len(contacts),
    "Key DOWN (raw)": len(downs),
    "Key UP (raw)": len(ups),

    "Key UP matched to completed contact": len(matched),

    "Matched contacts with exact same-key Key DOWN":
        int(matched["key_down_exact"].sum()),

    "Matched contacts with ANY exact Key DOWN":
        int(matched["any_key_down_exact"].sum()),

    "Matched contacts with same-key DOWN during contact":
        int(matched["key_down_inside"].sum()),

    "Matched contacts with same-key DOWN (+/- 10 ms)":
        int(matched["key_down_near_10ms"].sum()),

    "Matched contacts WITHOUT same-key DOWN (+/- 10 ms)":
        int((~matched["key_down_near_10ms"]).sum()),
})

display(summary.to_frame("Count"))

# 6. Analyze missing DOWN by key.
missing = matched[~matched["key_down_near_10ms"]].copy()

by_key = (
    missing.groupby("kid")
    .agg(
        missing_down=("ms", "size"),
        median_contact_ms=("duration_ms", "median"),
    )
    .sort_values("missing_down", ascending=False)
)

print("\nKeys with the most missing DOWN events:")
display(by_key.head(25))

# 7. Inspect individual contacts.
print("\nExamples of UP events without corresponding Key DOWN:")
display(
    missing[
        ["act", "kid", "down_ms", "up_ms",
         "duration_ms", "any_key_down_exact"]
    ].head(30)
)


,Count
Touch DOWN (raw),1759
Touch UP (raw),1757
Completed single-pointer contacts,1727
Key DOWN (raw),825
Key UP (raw),1682
Key UP matched to completed contact,1637
Matched contacts with exact same-key Key DOWN,803
Matched contacts with ANY exact Key DOWN,803
Matched contacts with same-key DOWN during contact,803
Matched contacts with same-key DOWN (+/- 10 ms),803



Keys with the most missing DOWN events:


,missing_down,median_contact_ms
kid,,
32,128,97.0
116,77,112.0
97,71,108.0
111,68,86.0
101,59,119.0
110,47,86.0
104,43,96.0
105,38,93.5
115,33,119.0



Examples of UP events without corresponding Key DOWN:


,act,kid,down_ms,up_ms,duration_ms,any_key_down_exact
2,918136191000001,100,1205018.0,1205164.0,146.0,False
4,918136191000001,110,1205742.0,1205859.0,117.0,False
5,918136191000001,116,1205968.0,1206076.0,108.0,False
8,918136191000001,-5,1206997.0,1207105.0,108.0,False
17,918136191000001,101,1209293.0,1209411.0,118.0,False
20,918136191000001,111,1209964.0,1210028.0,64.0,False
26,918136191000001,121,1217733.0,1217845.0,112.0,False
27,918136191000001,111,1218052.0,1218149.0,97.0,False
40,918136191000001,-5,1222806.0,1222871.0,65.0,False
44,918136191000001,111,1230300.0,1230396.0,96.0,False


In [8]:
import pandas as pd
from pathlib import Path

path = Path(
    "output_data/hmog_portrait_selection_v1_5/user_task_pairs.csv"
)

df = pd.read_csv(path, dtype={"user_id": str})

tasks = ["writing_sitting", "writing_walking"]
min_gestures = 500

# Підходять тільки учасники, у яких є 4 сесії
# з мінімум 500 придатними жестами в кожній.
eligible = df[
    df["task"].isin(tasks)
    & (df["distinct_session_numbers"] == 4)
    & (df["min_usable_in_session"] >= min_gestures)
]

sitting = set(
    eligible.loc[
        eligible["task"] == "writing_sitting", "user_id"
    ]
)

walking = set(
    eligible.loc[
        eligible["task"] == "writing_walking", "user_id"
    ]
)

common = sorted(sitting & walking)

print(f"Writing sitting: {len(sitting)}")
print(f"Writing walking: {len(walking)}")
print(f"Both activities: {len(common)}")

print("\nCommon user IDs:")
print(common)

# Експорт учасників для майбутнього експерименту
pd.DataFrame({"user_id": common}).to_csv(
    "hmog_common_writing_users.csv", index=False
)

Writing sitting: 77
Writing walking: 80
Both activities: 76

Common user IDs:
['100669', '151985', '171538', '180679', '186676', '207969', '240168', '248252', '256487', '257279', '261313', '264325', '278135', '326223', '336172', '342329', '352716', '366286', '368258', '389015', '395129', '396697', '398248', '405035', '431312', '472761', '525584', '526319', '527796', '538363', '539502', '540641', '553321', '554303', '561993', '578526', '579284', '588087', '622852', '657486', '675397', '698266', '710707', '717868', '720193', '745224', '763813', '771782', '776328', '777078', '785873', '785899', '808022', '815316', '827212', '856302', '856401', '862649', '863985', '865501', '865881', '872895', '879155', '892687', '893198', '893255', '897652', '913228', '923862', '937904', '962159', '966655', '973891', '984799', '986737', '990622']
